# 📓 Notebook 2: Building a chatbot agent with memory 🤖

In Notebook 1 we turned a PDF into a searchable vector database. Now we add the **brain**, an LLM, step by step:

```
LLM ──► + system prompt ──► + memory ──► + tool (PDF search) ──► 🤖 AGENT
```

## 1. Installation 📦
Run this once. After a fresh install, restart the kernel: **Kernel → Restart**.

In [ ]:
%pip install -q --extra-index-url https://download.pytorch.org/whl/cpu "torch==2.14.0" "sentence-transformers==6.1.0" "langchain-huggingface==1.2.2" "langchain==1.4.2" "langchain-core==1.6.5" "langchain-community==0.4.2" "langchain-text-splitters==1.1.2" "langgraph==1.2.12" "langchain-google-genai==4.4.0" "langchain-groq==1.1.3" "faiss-cpu==1.15.1" "pypdf==6.19.0" "python-dotenv==1.2.3"

## 2. Import the libraries 📚
| Library | What it does for us |
|---|---|
| `langchain_groq`, `langchain_google_genai` | talk to the Groq and Google Gemini LLMs |
| `langchain` | the `create_agent` function and the `@tool` decorator |
| `langgraph` | runs the agent step by step and gives it **memory** |
| `dotenv` | reads our secret API keys from the `.env` file |

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv("../.env")   # loads GROQ_API_KEY and GEMINI_API_KEY
print("✅ All libraries loaded!")

## 3. LLM model 🧠
An **LLM (Large Language Model)** is a huge neural network trained to predict the next word. We can use two free options:

| Provider | Model | Special thing |
|---|---|---|
| **Groq** | `llama-3.1-8b-instant` (Meta's Llama) | runs on special chips (LPUs) and is *extremely* fast |
| **Google** | `gemini-2.5-flash` | a powerful model from Google |

Pick one by changing `PROVIDER` below.

In [ ]:
PROVIDER = "groq"   # 👈 change to "gemini" to try Google Gemini

## 4. Model loading 🔌
We create the model object with our API key, then send our first message!

In [ ]:
if PROVIDER == "groq":
    api_key = os.getenv("GROQ_API_KEY") or getpass("Paste your GROQ_API_KEY: ")
    llm = ChatGroq(model="llama-3.1-8b-instant", api_key=api_key, temperature=0)
else:
    api_key = os.getenv("GEMINI_API_KEY") or getpass("Paste your GEMINI_API_KEY: ")
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=api_key, temperature=0)

response = llm.invoke("Explain what an LLM is to a first-year student in 2 sentences.")

print("🤖", response.text)
print("\n🔢 Tokens used:", response.usage_metadata)

💡 **Tokens** are small pieces of words. LLMs read and write tokens, and API usage is measured in tokens.

## 5. System prompt 🎭
A **system prompt** is a hidden instruction that tells the LLM **who it is** and **how to behave**.
Same question, different system prompt: watch the answer change!

In [ ]:
question = "What is a vector database?"

personalities = {
    "Teacher": "You are a patient teacher. Explain in 2 simple sentences with an everyday example.",
    "Pirate": "You are a pirate. Answer in 2 sentences, talking like a pirate.",
    "Poet": "You are a poet. Answer as a 4-line rhyming poem.",
}

for name, system_prompt in personalities.items():
    reply = llm.invoke([SystemMessage(system_prompt), HumanMessage(question)])
    print(f"🎭 {name}:\n{reply.text}\n")

## 6. Memory 💭
Surprise: **LLMs have no memory!** Every call starts from zero.
Let's prove it:

In [ ]:
llm.invoke("Hi! My name is Anu and I study Computer Science.")
reply = llm.invoke("What is my name?")
print("❌ Without memory:", reply.text)

The fix is simple: **send the whole conversation every time**. That list of messages *is* the memory.

In [ ]:
conversation = [SystemMessage("You are a friendly assistant. Keep answers short.")]


def chat(user_text):
    conversation.append(HumanMessage(user_text))
    reply = llm.invoke(conversation)   # the LLM sees ALL previous messages
    conversation.append(reply)
    return reply.text


chat("Hi! My name is Anu and I study Computer Science.")
print("✅ With memory:", chat("What is my name and what do I study?"))
print(f"\n📜 The memory now holds {len(conversation)} messages:")
for message in conversation:
    print(f"   {type(message).__name__:<14} {message.text[:70]}")

## 7. Give the LLM a tool: search our PDF 🔧
The LLM does not know our PDF. So we build the vector database from Notebook 1 and wrap the search in a **tool**, which is a Python function the LLM is allowed to call.
The text inside `"""..."""` (the docstring) tells the LLM **when** to use the tool.

In [ ]:
pages = PyPDFLoader("../data/sample_syllabus.pdf").load()
chunks = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150).split_documents(pages)
vector_db = FAISS.from_documents(chunks, HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"))
print(f"📚 PDF ready: {len(pages)} pages → {len(chunks)} chunks in FAISS")


@tool
def search_pdf(query: str) -> str:
    """Search the course syllabus PDF. Use it for questions about the course, modules, exams or marks."""
    print(f"   🔧 Tool called: search_pdf('{query}')")
    results = vector_db.similarity_search(query, k=3)
    return "\n\n".join(doc.page_content for doc in results)


print(search_pdf.invoke("exam duration")[:300], "...")

## 8. The Agent 🤖 = LLM + system prompt + tool + memory
An **agent** is an LLM that can **decide** what to do. It works in a loop called **ReAct** (*Reason + Act*):

```
question ──► 🧠 LLM thinks: "I need the PDF" ──► 🔧 calls search_pdf ──► 🧠 reads result ──► 💬 answers
```
* `create_agent` (from LangChain) builds this loop for us. Under the hood it is a **LangGraph** graph. *(Older tutorials use `create_react_agent` from `langgraph.prebuilt`: same idea, now renamed.)*
* `InMemorySaver` (from LangGraph) is the agent's **memory**. It stores each conversation under a `thread_id`.

In [ ]:
SYSTEM_PROMPT = """You are a friendly teaching assistant for engineering students.
Always use the search_pdf tool to look up facts before answering questions about the course.
If the PDF does not contain the answer, say so honestly. Keep answers short."""

memory = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=[search_pdf],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=memory,
)
print("✅ Agent created!")

Let's look at the agent's "brain": a LangGraph graph with a loop between the **model** and the **tools**. (Drawing the picture needs internet.)

In [ ]:
from IPython.display import Image, display

display(Image(agent.get_graph().draw_mermaid_png()))

## 9. Chat with the agent 💬
`pretty_print()` shows every step: your question, the tool call, what the tool returned, and the final answer.

In [ ]:
def ask(question, thread_id="student-1"):
    config = {"configurable": {"thread_id": thread_id}}   # thread_id = which conversation (memory) to use
    for step in agent.stream({"messages": [{"role": "user", "content": question}]}, config, stream_mode="values"):
        step["messages"][-1].pretty_print()


ask("How are the internal marks distributed?")

🧠 **Memory test:** this follow-up question does not say *what* "those marks" are. The agent has to remember!

In [ ]:
ask("And how many of those marks are for attendance?")

🆕 **A new thread = a new conversation.** With a different `thread_id`, the agent has no memory of the chat above:

In [ ]:
ask("What did I ask you before?", thread_id="student-2")

🚫 **Honesty test:** what does the agent do when the PDF has no answer?

In [ ]:
ask("Who won the 2022 football world cup according to the syllabus?")

## 🎉 You built an AI agent!
| Step | What you learned |
|---|---|
| LLM | a model that predicts text, used through an API |
| System prompt | controls the personality and the rules |
| Memory | LLMs forget, so we store and resend the conversation |
| Tool | a Python function the LLM can decide to call |
| Agent | LLM + tools + memory working in a loop (ReAct) |

This same agent powers our web app: `backend/core_rag.py` → FastAPI → Streamlit chat.

**Try it yourself:**
* Switch `PROVIDER` to `"gemini"`. Do the answers change?
* Change the `SYSTEM_PROMPT` so the agent always answers in Malayalam 😄
* Write a second tool, e.g. `calculator(expression: str)`, and add it to `tools=[...]`